In [10]:
import pandas as pd
from matplotlib import pyplot
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [11]:
load_dotenv()
database_url = os.getenv('DATABASE_URL')

engine = create_engine(database_url)

ranking_stdev_df = pd.read_sql('''
                SELECT
                    t.full_name AS team_name,
                    g.year,
                    STDDEV(g.ranking) AS ranking_stdev
                FROM (
                        SELECT
                            year,
                            id_home_team AS team_id,
                            home_team_ranking AS ranking
                        FROM
                            games
                        WHERE
                        home_team_ranking != -1
                    UNION ALL
                        SELECT
                            year,
                            id_away_team AS team_id,
                            away_team_ranking AS ranking
                        FROM
                            games
                        WHERE
                            away_team_ranking != -1
                ) AS g
                JOIN
                    teams AS t ON g.team_id = t.team_id
                GROUP BY
                    t.full_name,
                    g.year
                HAVING
                    COUNT(g.ranking) >= 2
                ORDER BY
                t.full_name,
                g.year;'''
                , engine)

print(ranking_stdev_df)

              team_name  year  ranking_stdev
0     Air Force Falcons  1958       2.768875
1     Air Force Falcons  1959       0.577350
2     Air Force Falcons  1969       0.577350
3     Air Force Falcons  1970       3.973396
4     Air Force Falcons  1971       1.414214
...                 ...   ...            ...
2705      Yale Bulldogs  1937       4.082483
2706      Yale Bulldogs  1944       1.414214
2707      Yale Bulldogs  1946       3.000000
2708      Yale Bulldogs  1959       4.242641
2709      Yale Bulldogs  1960       2.081666

[2710 rows x 3 columns]


In [12]:
top_five_teams_count_df = pd.read_sql('''
    SELECT
        t.full_name AS team_name,
        COUNT(*) as top_five_wins
    FROM
        (SELECT
            year,
            id_home_team AS team_id,
            home_team_ranking AS ranking
        FROM
            games
        WHERE
            home_team_ranking >= 1 AND home_team_ranking <=5
        UNION ALL
        SELECT
            year,
            id_away_team AS team_id,
            away_team_ranking AS ranking
        FROM
            games
        WHERE
            away_team_ranking >= 1 AND away_team_ranking <=5
        ) as g
    JOIN teams as t ON g.team_id = t.team_id
    GROUP BY
        t.full_name
    ORDER BY
        COUNT(*) DESC
                                      ''', engine)

print(top_five_teams_count_df)

                      team_name  top_five_wins
0          Alabama Crimson Tide            372
1              Oklahoma Sooners            327
2           Ohio State Buckeyes            318
3          Nebraska Cornhuskers            239
4   Southern California Trojans            232
..                          ...            ...
79      Saint Mary's (CA) Gaels              1
80                Yale Bulldogs              1
81       Tulsa Golden Hurricane              1
82               Duquesne Dukes              1
83         Oregon State Beavers              1

[84 rows x 2 columns]
